## Olist dataset

In [0]:
import os

import pyspark.sql.functions as f
import pyspark.sql.types as t

In [0]:
# CONSTANTES

CATALOG = 'big-data-spark-sql'
SCHEMA = 'olist'
VOLUME = f'/Volumes/{CATALOG}/{SCHEMA}/raw'

# PATHS
olist_customers = f"{VOLUME}/olist_customers_dataset.csv"
olist_geolocation = f"{VOLUME}/olist_geolocation_dataset.csv"
olist_orders = f"{VOLUME}/olist_orders_dataset.csv"
olist_order_items = f"{VOLUME}/olist_order_items_dataset.csv"
olist_order_payments = f"{VOLUME}/olist_order_payments_dataset.csv"
olist_order_reviews = f"{VOLUME}/olist_order_reviews_dataset.csv"
olist_products = f"{VOLUME}/olist_products_dataset.csv"
olist_product_category_translation = f"{VOLUME}/olist_product_category_translation.csv"
olist_sellers = f"{VOLUME}/olist_sellers_dataset.csv"

In [0]:
olist_customers_df = spark.read.csv(olist_customers, header=True, inferSchema=False)
olist_geolocation_df = spark.read.csv(olist_geolocation, header=True, inferSchema=False)
olist_orders_df = spark.read.csv(olist_orders, header=True, inferSchema=False)
olist_order_items_df = spark.read.csv(olist_order_items, header=True, inferSchema=False)
olist_order_payments_df = spark.read.csv(olist_order_payments, header=True, inferSchema=False)
olist_order_reviews_df = spark.read.csv(olist_order_reviews, header=True, inferSchema=False)
olist_products_df = spark.read.csv(olist_products, header=True, inferSchema=False)
olist_product_category_translation_df = spark.read.csv(olist_product_category_translation, header=True, inferSchema=False)
olist_sellers_df = spark.read.csv(olist_sellers, header=True, inferSchema=False)

In [0]:
olist_order_items_df.printSchema()

In [0]:
olist_order_items_df.display()

In [0]:
olist_order_items_df = (olist_order_items_df
 .withColumn('shipping_limit_date', f.to_timestamp(f.col('shipping_limit_date'), 'yyyy-MM-dd HH:mm:ss'))
)
olist_order_items_df.printSchema()

# withColumns é uma transformação que retorna um novo dataframe. Logo é necessário sobrescrever o dataframe original

In [0]:
olist_order_items_df.show()

In [0]:
olist_order_items_df = (olist_order_items_df
 .withColumn('price', f.col('price').cast(t.FloatType()))
 .withColumn('freight_value', f.col('price').cast('float'))
)
olist_order_items_df.printSchema()

### Escrita de arquivos

In [0]:
from pathlib import Path

OUTPUT_PATH = Path(VOLUME, 'parquet')

In [0]:
comments_messages_not_null = (olist_order_reviews_df
                 .filter(f.col('review_comment_message').isNotNull())
                 .select(
                     'review_id',
                     'order_id',
                     'review_comment_message'
                    )
                )

(comments_messages_not_null
 .repartition()
 .write
 .mode('overwrite')
 .parquet(str(Path(OUTPUT_PATH, 'write_example')))
)

In [0]:
# Ordenação

(olist_order_items_df
 .select('order_id', 'price')
 .groupBy('order_id')
 .agg(
     f.round(f.sum('price'), 2).alias('total_price')
 )
 .orderBy(f.col('total_price').desc())
 .display()
)